In [53]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_text_splitters import RecursiveCharacterTextSplitter

from qdrant_client import QdrantClient
from qdrant_client.http import models
from qdrant_client.models import PointStruct, Distance, VectorParams

load_dotenv()
HUGGINGFACEHUB_API_TOKEN=os.getenv("HUGGINGFACEHUB_API_TOKEN")
repo_id = "Qwen/Qwen2.5-72B-Instruct"

In [24]:
def chunking(path:str, n_chunk:int = 1000, chunk_overlap:int = 100):

    loader = PyPDFLoader(path)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = n_chunk,
        chunk_overlap = chunk_overlap
    )

    documents = loader.load()
    chunks = splitter.split_documents(documents)
    chunks = [chunk.page_content for chunk in chunks]
    return chunks


path = "mongodb.pdf"
chunks = chunking(path=path)

In [26]:
import sys
sys.getsizeof(chunks)

920

In [46]:
def get_embeddings(chunks:list):

    hf_embed = HuggingFaceEmbeddings()
    embedded_chunks = hf_embed.embed_documents(chunks)
    return embedded_chunks, hf_embed

In [47]:
embedded_chunks, hf_embed = get_embeddings(chunks=chunks)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [48]:
qd_client.delete_collection(collection_name=COLLECTION_NAME)

client = QdrantClient(
        host = "localhost",
        port = 6333
    )
client.create_collection(
    collection_name = "my_collection",
    vectors_config=VectorParams(size=768, distance=Distance.DOT),
)

True

In [49]:
COLLECTION_NAME = "my_collection"
BATCH_SIZE = 100  # Tune this down if chunks are very large
qd_client = QdrantClient(
        host = "localhost",
        port = 6333
    )


def store_embeddings2qdrant(embedded_chunks:list):

    for batch_start in range(0, len(embedded_chunks), BATCH_SIZE):
        batch_end = batch_start + BATCH_SIZE
        batch_points = [
            PointStruct(
                id = i,
                vector= embedded_chunks[i],
                payload= {"text" : chunks[i]}
            )
            for i in range(batch_start, min(batch_end, len(embedded_chunks)))
        ]

        qd_client.upsert(
            collection_name=COLLECTION_NAME,
            wait = True,
            points= batch_points
        )
        print(f"Upserted batch {batch_start}–{min(batch_end, len(embedded_chunks))-1}")

    print(f"embedding upserted successfully")

In [50]:
store_embeddings2qdrant(embedded_chunks=embedded_chunks)

Upserted batch 0–96
embedding upserted successfully


In [51]:
def get_context(query:str):

    query_embed = hf_embed.embed_query(query)
    data = client.query_points(
        collection_name="my_collection",
        query=query_embed,
        with_payload=True,
        limit=3
    )
    context = "\n\n".join([payload.payload.get("text") for payload in data.points])
    print(f"context: {context}")
    return context

In [52]:
get_context("what problem mongodb solves? which is not solved by mysql?")

context: In This Chapter
The message from this chapter is that MongoDB, in most cases, can replace a relational database. It’s much simpler and
straightforward; it’s faster and generally imposes fewer restrictions on application developers. The lack of transactions
can be a legitimate and serious concern. However, when people ask where does MongoDB sit with respect to the new
data storage landscape? the answer is simple: right in the middle .
48

Tools and Maturity
You probably already know the answer to this, but MongoDB is obviously younger than most relational database
systems. This is absolutely something you should consider, though how much it matters depends on what you are
doing and how you are doing it. Nevertheless, an honest assessment simply can’t ignore the fact that MongoDB
is younger and the available tooling around isn’t great (although the tooling around a lot of very mature relational
databases is pretty horrible too!). As an example, the lack of support for base-10 fl

'In This Chapter\nThe message from this chapter is that MongoDB, in most cases, can replace a relational database. It’s much simpler and\nstraightforward; it’s faster and generally imposes fewer restrictions on application developers. The lack of transactions\ncan be a legitimate and serious concern. However, when people ask where does MongoDB sit with respect to the new\ndata storage landscape? the answer is simple: right in the middle .\n48\n\nTools and Maturity\nYou probably already know the answer to this, but MongoDB is obviously younger than most relational database\nsystems. This is absolutely something you should consider, though how much it matters depends on what you are\ndoing and how you are doing it. Nevertheless, an honest assessment simply can’t ignore the fact that MongoDB\nis younger and the available tooling around isn’t great (although the tooling around a lot of very mature relational\ndatabases is pretty horrible too!). As an example, the lack of support for base-1

In [ ]:
def ask_llm(query:str, context:str = None):

    endpoint = HuggingFaceEndpoint(
            huggingfacehub_api_token = HUGGINGFACEHUB_API_TOKEN,
            repo_id = repo_id,
            temperature = 0.5,
            max_new_tokens = 1024
        )
    
    llm = ChatHuggingFace(llm=endpoint)
    response = llm.invoke(query)
    return response.content


# print(ask_llm("generate a poem on sex"))